In [1]:
import numpy as np

rng = np.random.default_rng(0)
n = 1000
sqft = rng.normal(2000, 500, n)
age = rng.uniform(0, 50, n)

TRUE_EFFECT = 0.1  # $k per sq ft
price = 50 + TRUE_EFFECT * sqft + 100 * np.exp(-age / 8) + rng.normal(0, 5, n)

X = np.column_stack([sqft, age])

# 75/25 train-test split
idx = rng.permutation(n)
tr, te = idx[:750], idx[750:]
X_tr, X_te, y_tr, y_te = X[tr], X[te], price[tr], price[te]

# OLS via least squares
def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])

beta, *_ = np.linalg.lstsq(add_intercept(X_tr), y_tr, rcond=None)
ols_predict = lambda X: add_intercept(X) @ beta

# k-nearest neighbors (flexible model)
mu, sd = X_tr.mean(axis=0), X_tr.std(axis=0)

def knn_predict(X, k=10):
    Z, Z_tr = (X - mu) / sd, (X_tr - mu) / sd
    dists = ((Z[:, None, :] - Z_tr[None, :, :]) ** 2).sum(axis=2)
    nearest = np.argsort(dists, axis=1)[:, :k]
    return y_tr[nearest].mean(axis=1)

def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))

# Prediction score
print(f"OLS test RMSE: {rmse(y_te, ols_predict(X_te)):.2f}")
print(f"KNN test RMSE: {rmse(y_te, knn_predict(X_te)):.2f}")

# Inference score: estimated effect of +1 sq ft
bump = X_te + np.array([100, 0])
ols_effect = beta[1]
knn_effect = (knn_predict(bump) - knn_predict(X_te)).mean() / 100
print(f"True effect: {TRUE_EFFECT}, OLS: {ols_effect:.3f}, KNN: {knn_effect:.3f}")

OLS test RMSE: 16.99
KNN test RMSE: 8.18
True effect: 0.1, OLS: 0.101, KNN: 0.096
